In [1]:
import pandas as pd
#数据导入
data = pd.read_csv('D:/JupyterNoteBook/portfolio/Project2/data/订单表.csv', encoding='GBK')
data.head()

,订单id,店铺id,日期,订单状态,流量渠道,数量,折扣率,明细金额,成本价快照,用户id,佣金率,商品id,商品成交单价,总成本
0,O0000233,st003,2024/1/1 0:00,已完成,自然搜索,1,1,158,90.93,JD-U020669,0.052,p0000010,158,90.93
1,O0000235,st003,2024/1/1 0:00,已完成,自然搜索,1,1,158,90.93,JD-U020643,0.052,p0000010,158,90.93
2,O0000240,st003,2024/1/1 0:00,已完成,自然搜索,1,1,158,90.93,JD-U026303,0.052,p0000010,158,90.93
3,O0000248,st003,2024/1/1 0:00,已完成,自然搜索,1,1,158,90.93,JD-U033270,0.052,p0000010,158,90.93
4,O0000608,st003,2024/1/2 0:00,已完成,自然搜索,1,1,158,90.93,JD-U037235,0.052,p0000010,158,90.93


In [2]:
#数据描述
data.shape

(472481, 14)

In [3]:
data.describe()

,数量,折扣率,明细金额,成本价快照,佣金率,商品成交单价,总成本
count,472481.000000,472481.000000,472481.000000,472481.000000,472481.000000,472481.000000,472481.000000
mean,1.442507,0.920223,179.575816,80.519184,0.047320,132.145701,109.038142
std,1.124797,0.270948,222.898763,106.361281,0.017866,176.987675,133.124334
min,1.000000,0.000000,13.000000,13.130000,0.016000,13.000000,13.130000
25%,1.000000,1.000000,74.000000,45.130000,0.020000,67.000000,50.700000
50%,1.000000,1.000000,128.000000,59.510000,0.055000,110.000000,70.230000
75%,1.000000,1.000000,175.000000,87.520000,0.060000,145.000000,105.550000
max,18.000000,1.000000,4156.000000,1323.090000,0.070000,2078.000000,2646.180000


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 472481 entries, 0 to 472480
Data columns (total 14 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   订单id    472481 non-null  object 
 1   店铺id    472481 non-null  object 
 2   日期      472481 non-null  object 
 3   订单状态    472481 non-null  object 
 4   流量渠道    472481 non-null  object 
 5   数量      472481 non-null  int64  
 6   折扣率     472481 non-null  int64  
 7   明细金额    472481 non-null  int64  
 8   成本价快照   472481 non-null  float64
 9   用户id    454466 non-null  object 
 10  佣金率     472481 non-null  float64
 11  商品id    472481 non-null  object 
 12  商品成交单价  472481 non-null  int64  
 13  总成本     472481 non-null  float64
dtypes: float64(3), int64(4), object(7)
memory usage: 50.5+ MB


In [5]:
#查看缺失值
data.isnull().sum()

订单id          0
店铺id          0
日期            0
订单状态          0
流量渠道          0
数量            0
折扣率           0
明细金额          0
成本价快照         0
用户id      18015
佣金率           0
商品id          0
商品成交单价        0
总成本           0
dtype: int64

In [6]:
#删掉无用户标识的行（18,015 行：已取消 17,996 + 待付款 19）
#注意：这≠「删未成交订单」。已取消/已退款/已发货中带 用户id 的行仍在表内，
#      业务口径「成交 = 订单状态已完成」由口径层施加，见 sql/00_schema.sql 的 v_成交明细
data = data.dropna(subset = ['用户id'])

In [7]:
data.isnull().sum().sum()

np.int64(0)

In [8]:
#查看重复值
data.duplicated().sum()

np.int64(0)

## 字段校验

把 `doc/数据说明.md`「4：数据质量」的结论写成断言，任一不过即中断。


In [9]:
#字段校验
raw = pd.read_csv('D:/JupyterNoteBook/portfolio/Project2/data/订单表.csv', encoding='GBK')

#缺失值：只有 用户id 缺失，且都是未成交订单
na = raw['用户id'].isna()
assert len(raw) == 472481 and raw.shape[1] == 14
assert int(na.sum()) == 18015
na_status = raw.loc[na, '订单状态'].value_counts()
assert int(na_status['已取消']) == 17996 and int(na_status['待付款']) == 19
assert raw.drop(columns=['用户id']).isnull().sum().sum() == 0
assert data.isnull().sum().sum() == 0

#重复值
assert raw.duplicated().sum() == 0
assert raw.duplicated(subset=['订单id', '商品id']).sum() == 0

#总成本 = 成本价快照 × 数量，全部成立
assert (raw['总成本'].round(2) == (raw['成本价快照'] * raw['数量']).round(2)).all()

#明细金额 = 商品成交单价 × 数量 × 折扣率，只有 81.84% 的行成立，其余含未记录优惠
assert int((raw['明细金额'] == raw['商品成交单价'] * raw['数量'] * raw['折扣率']).sum()) == 386668

#归属：一个用户只在一个店铺，一个订单只对一个店铺一个用户
assert (raw.dropna(subset=['用户id']).groupby('用户id')['店铺id'].nunique() == 1).all()
order = raw.groupby('订单id').agg(s=('店铺id', 'nunique'), u=('用户id', 'nunique'))
assert (order['s'] > 1).sum() + (order['u'] > 1).sum() == 0

#枚举与数值范围
assert sorted(raw['订单状态'].unique()) == sorted(['已完成', '已取消', '已退款', '已发货', '待发货', '待付款'])
assert sorted(map(int, raw['折扣率'].unique())) == [0, 1]
assert raw['店铺id'].nunique() == 6 and raw['商品id'].nunique() == 1320 and raw['流量渠道'].nunique() == 9
assert int(raw['数量'].min()) == 1 and int(raw['数量'].max()) == 18
assert int(raw['明细金额'].min()) == 13 and int(raw['明细金额'].max()) == 4156
assert (raw['明细金额'] > 0).all() and (raw['成本价快照'] > 0).all() and (raw['总成本'] > 0).all()
assert raw['佣金率'].nunique() == 12

print('字段校验通过')


字段校验通过


In [10]:
data.to_csv('D:/JupyterNoteBook/portfolio/Project2/data/订单表_clear.csv',index = False)
print('清洗完成')

清洗完成
